In [1]:
%pip install langchain huggingface_hub sentence_transformers faiss-cpu unstructured chromadb Cython tiktoken unstructured[local-inference] pdf2image openai  groq -q

Note: you may need to restart the kernel to use updated packages.


In [17]:
%pip install mistral_inference


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\Cantt Computer\\AppData\\Local\\Temp\\pip-install-g9wdm6t6\\xformers_1d5d8a44152d43e680910d12a6f1d35a\\third_party\\flash-attention\\csrc\\composable_kernel\\client_example\\24_grouped_conv_activation\\grouped_convnd_bwd_data_bilinear\\grouped_conv_bwd_data_bilinear_residual_fp16.cpp'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths




  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ---------------------------------------- 0.0/7.8 MB ? eta -:--:--
     ----------------------------- ---------- 5.8/7.8 MB 29.3 MB/s eta 0:00:01
     ---------------------------------------- 7.8/7.8 MB 24.0 MB/s eta 0:00:00


In [2]:
from langchain.embeddings import HuggingFaceEmbeddings
# Text Splitter
from langchain.text_splitter import CharacterTextSplitter
from langchain.document_loaders import UnstructuredPDFLoader
from langchain.indexes import VectorstoreIndexCreator
from langchain.vectorstores import Chroma

e:\IDEs-tools\anaconda\envs\rag-env\lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in HuggingFaceInferenceAPIEmbeddings has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [3]:
import os
pdf_folder_path = './archive/data/data/ACCOUNTANT/'
os.listdir(pdf_folder_path)

['10554236.pdf',
 '10674770.pdf',
 '11163645.pdf',
 '11759079.pdf',
 '12065211.pdf',
 '12202337.pdf',
 '12338274.pdf',
 '12442909.pdf',
 '12780508.pdf',
 '12802330.pdf',
 '13072019.pdf',
 '13130984.pdf',
 '13294301.pdf',
 '13491889.pdf',
 '13701259.pdf',
 '14055988.pdf',
 '14126433.pdf',
 '14224370.pdf',
 '14449423.pdf',
 '14470533.pdf',
 '14491649.pdf',
 '14496667.pdf',
 '15289348.pdf',
 '15363277.pdf',
 '15592167.pdf',
 '15821633.pdf',
 '15906625.pdf',
 '16237710.pdf',
 '17306905.pdf',
 '17407184.pdf',
 '17556527.pdf',
 '18132924.pdf',
 '18365791.pdf',
 '18569929.pdf',
 '18635654.pdf',
 '18669563.pdf',
 '19446337.pdf',
 '19545827.pdf',
 '20082776.pdf',
 '20253563.pdf',
 '20345168.pdf',
 '20393721.pdf',
 '20624984.pdf',
 '21031285.pdf',
 '21338490.pdf',
 '21763056.pdf',
 '21794875.pdf',
 '21853199.pdf',
 '22465498.pdf',
 '22925443.pdf',
 '23139819.pdf',
 '23246831.pdf',
 '23387174.pdf',
 '23416654.pdf',
 '23438112.pdf',
 '23513618.pdf',
 '23636277.pdf',
 '23734441.pdf',
 '24103168.pdf

In [4]:
import os
import shutil

# Define the root folder and the destination folder
root_folder = "./archive/data/data"
destination_folder = os.path.join(root_folder, "all_categories")

# Create the destination folder if it doesn't exist
if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)

# Traverse through all subdirectories in the root folder
for foldername, subfolders, filenames in os.walk(root_folder):
    for filename in filenames:
        # Check if the file is a PDF
        if filename.endswith('.pdf'):
            # Get full path of the source and destination file
            source_file = os.path.join(foldername, filename)
            destination_file = os.path.join(destination_folder, filename)

            # Copy the PDF file to the destination folder, skip if it's the same file
            if not os.path.exists(destination_file):
                shutil.copy2(source_file, destination_file)

print(f"All PDFs have been copied to {destination_folder}.")

All PDFs have been copied to ./archive/data/data\all_categories.


In [5]:
loaders = [UnstructuredPDFLoader(os.path.join(destination_folder, fn)) for fn in os.listdir(destination_folder)]
loaders

 ...]

In [6]:
fil = {"persist_directory": "E:\\SynergyTech\\chat-assistant\\ResumeScan-Agent\\vector_db\\"}
vector_store = VectorstoreIndexCreator(
    embedding=HuggingFaceEmbeddings(),
    text_splitter=CharacterTextSplitter(chunk_size=1000, chunk_overlap=0), 
    vectorstore_kwargs=fil).from_loaders(loaders)

C:\Users\Cantt Computer\AppData\Local\Temp\ipykernel_14816\2920725003.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding=HuggingFaceEmbeddings(),
C:\Users\Cantt Computer\AppData\Local\Temp\ipykernel_14816\2920725003.py:3: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding=HuggingFaceEmbeddings(),
e:\IDEs-tools\anaconda\envs\rag-env\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https:/

In [9]:
from langchain_community.llms import HuggingFaceEndpoint
from langchain.chains.question_answering import load_qa_chain
from langchain import HuggingFaceHub

In [10]:
llm=HuggingFaceEndpoint(repo_id="mistralai/Mistral-7B-Instruct-v0.2", temperature=0.1, max_length=512)

WARNING! max_length is not default parameter.
                    max_length was transferred to model_kwargs.
                    Please make sure that max_length is what you intended.


In [12]:
from langchain.chains import RetrievalQA
chain = RetrievalQA.from_chain_type(llm=llm,
                                    chain_type="stuff",
                                    retriever=vector_store.vectorstore.as_retriever(),
                                    input_key="question")

In [16]:
chain.run('What is the highest amount of experience among Accountant candidates?')

HfHubHTTPError: 429 Client Error: Too Many Requests for url: https://api-inference.huggingface.co/models/mistralai/Mistral-7B-Instruct-v0.2 (Request ID: HPaleWehM4ppitOPV_eek)

Please log in or use a HF access token

In [6]:
from langchain.vectorstores import Chroma
import chromadb
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.document_loaders import PyPDFLoader


def load_chunk_persist_pdf(path: str) -> Chroma:
    pdf_folder_path = path
    documents = []
    for file in os.listdir(pdf_folder_path):
        if file.endswith('.pdf'):
            pdf_path = os.path.join(pdf_folder_path, file)
            loader = PyPDFLoader(pdf_path)
            documents.extend(loader.load())
    text_splitter = CharacterTextSplitter(chunk_size=512, chunk_overlap=10)
    chunked_documents = text_splitter.split_documents(documents)
    client = chromadb.Client()
    if client.list_collections():
        consent_collection = client.create_collection("consent_collection")
    else:
        print("Collection already exists")
    vectordb = Chroma.from_documents(
        documents=chunked_documents,
        embedding=HuggingFaceEmbeddings(),
        persist_directory="E:\\SynergyTech\\chat-assistant\\ResumeScan-Agent\\vector_db\\"
    )
    vectordb.persist()
    print("Chroma DB created")
    return vectordb

In [7]:
vector_db = load_chunk_persist_pdf(pdf_folder_path)

Collection already exists


C:\Users\Cantt Computer\AppData\Local\Temp\ipykernel_26656\1964119847.py:25: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding=HuggingFaceEmbeddings(),
C:\Users\Cantt Computer\AppData\Local\Temp\ipykernel_26656\1964119847.py:25: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding=HuggingFaceEmbeddings(),
e:\IDEs-tools\anaconda\envs\rag-env\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https

Chroma DB created


C:\Users\Cantt Computer\AppData\Local\Temp\ipykernel_26656\1964119847.py:28: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


 ### GPT4All as below

In [8]:
from langchain_community.llms import GPT4All
#from gpt4all import GPT4All

llm = GPT4All(
    model="orca-mini-3b-gguf2-q4_0.gguf",
    max_tokens=2048, allow_download = True
)

### LLM response with RAG

In [9]:
from langchain import hub
# retrieve relevant docs
rag_prompt = hub.pull("rlm/rag-prompt")
retriever = vector_db.as_retriever()
rag_prompt.messages

e:\IDEs-tools\anaconda\envs\rag-env\lib\site-packages\langsmith\client.py:322: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(
e:\IDEs-tools\anaconda\envs\rag-env\lib\site-packages\langsmith\client.py:5301: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  prompt = loads(json.dumps(prompt_object.manifest))


[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]

In [10]:
def format_documents(documents):
    return "\n\n".join(doc.page_content for doc in documents)

In [11]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Create the langchain with retriever,
# prompt template and LLM
qa_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

In [12]:
qa_chain.invoke("Give me highest number of experience in candidate for Accounting? Summarize in 50 words.")

'ERROR: The prompt size exceeds the context window size and cannot be processed.'

In [2]:
# for preprocessing html data  
!pip install beautifulsoup4

# For RAG
!pip install langchain
!pip install langchainhub
!pip install chromadb
!pip install gpt4all

!pip install tqdm

# Needed if using LlamaCpp from LangChain
# !pip install llama-cpp-python


In [3]:
from tqdm import tqdm

# To deal with emails 
import email
from email.policy import default
from bs4 import BeautifulSoup

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import GPT4AllEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import LlamaCpp
from langchain_core.output_parsers import StrOutputParser
from langchain.docstore.document import Document
from langchain import hub
from langchain_core.runnables import RunnablePassthrough

e:\IDEs-tools\anaconda\envs\rag-env\lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in GPT4AllEmbeddings has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


# Preprocess the Promotion Emails

In [6]:
# code from Stack Overflow:
# https://stackoverflow.com/questions/59681461/read-a-big-mbox-file-with-python
class MboxReader:
    def __init__(self, filename):
        self.handle = open(filename, 'rb')
        assert self.handle.readline().startswith(b'From ')

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, exc_traceback):
        self.handle.close()

    def __iter__(self):
        return iter(self.__next__())

    def __next__(self):
        lines = []
        while True:
            line = self.handle.readline()
            if line == b'' or line.startswith(b'From '):
                yield email.message_from_bytes(b''.join(lines), policy=default)
                if line == b'':
                    break
                lines = []
                continue
            lines.append(line)

In [7]:
path = "./Takeout/Mail/Category_promotions.mbox"
mbox = MboxReader(path)
emails_to_process = 10

current_mails = 0
promo_contents = ""
for __, message in tqdm(enumerate(mbox)):
    payload = message.get_payload(decode=True)
    if payload:
        current_mails += 1
        if current_mails > emails_to_process:
            break
        soup = BeautifulSoup(payload, 'html.parser')
        body_text = soup.get_text().replace('"','').replace("\n", "").replace("\t", "").strip()
        promo_contents += body_text + " "

13it [00:00, 20.89it/s]


In [8]:
promo_contents

'autoTRADER | autoHEBDOCould this be your next car? Check it out at AutoTrader \r            Find out what your car is worth instantly.\xa0\xa0Check  NowNew Price Drops for New & Used 2017 - 2024 Cars for sale in Toronto 32019 Chevrolet Cruze Mississauga, ON\xa0  \r59,000 km\xa0 Mississauga, ON\r$11,000View2019 Nissan LEAF Mississauga, ON\xa0  \r49,300 km\xa0 Mississauga, ON\r$12,500View Your Search Criteria:\xa0New & Used 2017 - 2024 Cars for sale in Toronto 3\r\xa0\rMake:Any\rLocation:Toronto, ON\rModel:Any\rSearch Radius:50 km \r\xa0View ListingsDo more on AutoTrader\r    \xa0Latest Reviews and Advice\r            \xa0Check out our editorial page for all of the latest information, advice, and expert reviews on the world of vehicles!\xa0\r            \xa0View Now\r                \xa0\r \xa0\r    \xa0Find Out What Your Car is Worth\xa0There is no better time to sell your car! Use Instant Cash Offer for an accurate trade-in value for your vehicle. Just enter your vehicle details and i

In [9]:
# write the arxiv emails into a txt file
with open("promo_contents.txt", "w", encoding="utf-8") as f:
 f.write(promo_contents)

### convert text document to langchain document format

In [10]:
doc = Document(page_content=promo_contents, 
                metadata={"source": "local"})

# split into different chunks
# chunk_size and chunk_overlap are a hyperparameters we choose 
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)

# split the documents and convert to vector stores
all_splits = text_splitter.split_documents([doc])
vector_store = Chroma.from_documents(documents=all_splits, 
                                     embedding=GPT4AllEmbeddings())

### If using fine-tuned model, quantize to GGUF Format
For fine-tuning a given LLM, checkout [this video](https://youtu.be/_xxGMSVLwU8?feature=shared)

For quantizing, we can either: 
- Use the llama.cpp library written in C,C++ for this. Checkout [this video](https://youtu.be/j7ahltwlFH0?feature=shared)
- Or we can use LangChain's functionality. 

In any case we need the model to be converted to `gguf` format to run on the CPU.

## Load the quantize LLM model 
For the quantized model, we can either, 
- Use the LlamaCpp class from LangChain
- Use the GPT4All library

In any case, the models need to be quantized models

### Either use LlamaCpp as below

In [62]:
n_gpu_layers = 1 

n_batch = 512
quantized_gguf_model = "../generative-ai-course/quantized_models/ft-Q8_K_M.gguf"

# Initiate the LlamaCpp class to run the LLM
llm = LlamaCpp(
    model_path=quantized_gguf_model,
    n_gpu_layers=n_gpu_layers,
    n_batch=n_batch,
    n_ctx=1024,
    f16_kv=True,
    verbose=True,
)

ImportError: Could not import llama-cpp-python library. Please install the llama-cpp-python library to use this embedding model: pip install llama-cpp-python

### Or use GPT4All as below

In [11]:
from langchain_community.llms import GPT4All
#from gpt4all import GPT4All

llm = GPT4All(
    model="orca-mini-3b-gguf2-q4_0.gguf",
    max_tokens=2048, allow_download = True
)
# model = GPT4All(
#     "orca-mini-3b-gguf2-q4_0.gguf"
# )

#llm.invoke("what is Retrieval Augmented Generation?")

In [12]:
llm.invoke("What is the price of mazda?")

'\nWhat is the price of Mazda3?'

## Create the LangChain with and without RAG
For a given prompt, 
- Create a langchain without retrieval and see the response
- Create a langchain with the retrieval object

And see how the response differs

### LLM response without RAG

In [13]:
# retrieve relevant docs
rag_prompt = hub.pull("rlm/rag-prompt")
retriever = vector_store.as_retriever()

# Create the langchain with retriever
qa_chain = (
    {"context": {}, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)
qa_chain.invoke("what is the value of a mazda3?")

e:\IDEs-tools\anaconda\envs\rag-env\lib\site-packages\langsmith\client.py:322: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(
e:\IDEs-tools\anaconda\envs\rag-env\lib\site-packages\langsmith\client.py:5301: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  prompt = loads(json.dumps(prompt_object.manifest))


' I am sorry, but I do not have enough information to provide an accurate answer to your question. Can you please provide me with more context or specific details about the Mazda3 that you are referring to?'

### LLM response with RAG

In [14]:
# retrieve relevant docs
rag_prompt = hub.pull("rlm/rag-prompt")
rag_prompt.messages

e:\IDEs-tools\anaconda\envs\rag-env\lib\site-packages\langsmith\client.py:322: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]

In [15]:
def format_documents(documents):
    return "\n\n".join(doc.page_content for doc in documents)

In [16]:
retriever = vector_store.as_retriever()

# Create the langchain with retriever,
# prompt template and LLM
qa_chain = (
    {"context": retriever | format_documents, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)
qa_chain.invoke("what is the average price of Mazda3?")

' The average price of Mazda3 is $19,500.'